In [1]:
import sys
#lets you access python's runtime environment
from pathlib import Path
#sys.path is a built in variable in the sys module and contains a list of directories that is seached through when you do an import
#so we are appending the src directory to that
sys.path.append(str(Path().resolve().parent / "src"))
import config

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score

panel_data_path = config.PROJECT_ROOT/ "data" / "panel_data.csv"
panel_data = pd.read_csv(panel_data_path)

Train/test split must be time-aware. Entries in the train set should all predate entries in the test set.

In [2]:
year_counts = panel_data.groupby('Year').size().reset_index(name = 'NumberOfRows')
year_counts = year_counts.sort_values('Year')
year_counts['Cumulative'] = year_counts['NumberOfRows'].cumsum()
total_rows = year_counts.loc[len(year_counts)-1, 'Cumulative']
year_counts['EntriesPrecedingAndIncluding(%)'] = (year_counts['Cumulative']/total_rows) *100
year_counts

,Year,NumberOfRows,Cumulative,EntriesPrecedingAndIncluding(%)
0,2015,20521,20521,6.723501
1,2016,26591,47112,15.435778
2,2017,31939,79051,25.900273
3,2018,37641,116692,38.232972
4,2019,40916,157608,51.638692
5,2020,17673,175281,57.429074
6,2021,25873,201154,65.906105
7,2022,27153,228307,74.802515
8,2023,35947,264254,86.580192
9,2024,40959,305213,100.000000


Could take entries from 2023 and before for train set for approximately 87/13 train/test split. 

In [3]:


panel = panel_data.drop(columns = ['Name', 'WeightClassKg', 'Sex'])
train = panel[panel['Year']<=2023].copy()
test = panel[panel['Year']>2023].copy()

train_X = train.drop(columns = 'CompetesNextYear')
train_y = train['CompetesNextYear']
test_X = test.drop(columns = 'CompetesNextYear')
test_y = test['CompetesNextYear']



clf = RandomForestClassifier()
clf.fit(train_X, train_y)



,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

In [4]:

preds = clf.predict(test_X)

acc = accuracy_score(test_y,preds)
f1 = f1_score(test_y, preds)

In [5]:
panel_data.groupby('CompetesNextYear').size().reset_index(name = 'Number')

,CompetesNextYear,Number
0,False,180574
1,True,124639


In [6]:
competes_multiple = panel_data.groupby('Name').size().reset_index(name = 'YearsCompeted')

In [7]:
print(acc)
print(f1)

0.6228667692082326
0.6027312707352828


In [8]:
len(panel_data[panel_data['NumberOfMeets']>1])/len(panel_data)

0.34208569097646563

In [9]:
panel

,Year,Age,TimeCompetingYearEnd,NumberOfMeets,Goodlift,Dots,Wilks,BestMeetOfYear,AverageAttemptsMade,MostAttemptsMade,LeastAttemptsMade,LastMeetAttemptsMade,MeetsSinceLastBombOut,NumberOfBombOuts,PercentageImprovement,CompetesNextYear,WeightClassKgCode,F,M,Mx
0,2017,16.5,27,1,61.15,300.75,295.29,300.0,8.0,8,8,8,999,0,NaN,False,0.714286,1,0,0
1,2019,17.5,91,1,36.81,181.55,180.52,250.0,7.0,7,7,7,999,0,NaN,False,0.375000,0,1,0
2,2022,22.5,143,1,76.21,386.08,388.02,480.0,7.0,7,7,7,999,0,NaN,False,0.250000,0,1,0
3,2019,16.0,96,1,45.30,221.93,219.95,317.5,7.0,7,7,7,999,0,NaN,False,0.500000,0,1,0
4,2016,27.5,89,1,NaN,NaN,NaN,NaN,0.0,0,0,0,0,1,NaN,False,0.375000,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
305208,2024,NaN,276,1,75.72,369.43,365.36,545.0,6.0,6,6,6,999,0,NaN,False,0.500000,0,1,0
305209,2024,NaN,276,1,62.03,302.67,306.93,262.5,6.0,6,6,6,999,0,NaN,False,0.375000,1,0,0
305210,2024,NaN,276,1,55.06,268.19,272.53,230.0,3.0,3,3,3,999,0,NaN,False,0.375000,1,0,0
305211,2024,NaN,276,1,46.00,225.19,227.26,200.0,4.0,4,4,4,999,0,NaN,False,0.500000,1,0,0


In [10]:
importances = clf.feature_importances_

feature_importance_df = pd.DataFrame({
    'Feature': train_X.columns,
    'Importance': importances
}).sort_values('Importance', ascending=False, ignore_index =True)

print(feature_importance_df)

                  Feature    Importance
0    TimeCompetingYearEnd  1.400086e-01
1                   Wilks  1.148440e-01
2                Goodlift  1.145689e-01
3                    Dots  1.141414e-01
4                     Age  1.133591e-01
5          BestMeetOfYear  1.002426e-01
6                    Year  6.861247e-02
7   PercentageImprovement  6.229579e-02
8       WeightClassKgCode  4.659025e-02
9           NumberOfMeets  3.103759e-02
10    AverageAttemptsMade  2.342231e-02
11      LeastAttemptsMade  1.848891e-02
12   LastMeetAttemptsMade  1.826936e-02
13       MostAttemptsMade  1.751416e-02
14  MeetsSinceLastBombOut  4.752377e-03
15                      F  4.188111e-03
16                      M  4.011654e-03
17       NumberOfBombOuts  3.652147e-03
18                     Mx  3.563177e-07
